# 0: Install & Mount (run once per Colab session)

In [ ]:
!python -V

Python 3.12.12


In [ ]:
import os

if "COLAB_GPU" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive"
else:
    BASE_DIR = "/home/infopro3/Documents"


Mounted at /content/drive


In [ ]:
%%capture
# ===============================
# ENVIRONMENT & DEPENDENCIES
# ===============================
import os, re, torch

# Detect Colab
IS_COLAB = "COLAB_" in "".join(os.environ.keys())

if not IS_COLAB:
    # Non-Colab environment
    !pip install unsloth
else:
    # ---- Colab-specific install (Unsloth recommended) ----
    v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")

    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

# Pinned versions (as per your code)
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

# ===============================
# MODEL IMPORT & BASE CONFIG
# ===============================
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None            # auto-detect (fp16 for T4)
load_in_4bit = True     # memory efficient


In [ ]:
!ls /content/drive/MyDrive/jawiverse/nirnama/dataset/


jawi_rumi_parallel_v1.xlsx  jawi_rumi_v1.xlsx


# 1: Load model + LoRA

In [ ]:
# ===============================
# LOAD BASE MODEL
# ===============================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# ===============================
# APPLY LoRA (UNCHANGED LOGIC)
# ===============================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("✅ Environment & model ready")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Environment & model ready


# 2: DATA PREP + STANDARDIZE + LOAD

In [ ]:
# ===============================
# DATA PREP + STANDARDIZE + LOAD (ONE CELL)
# ===============================
import pandas as pd
import json
import os
import re
from datasets import load_dataset
from unsloth.chat_templates import (
    get_chat_template,
    standardize_sharegpt,
)

# ---------- TEXT CLEANING ----------
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = re.sub(r'[\u202a-\u202e\u200e\u200f]', '', str(text))
    return text.strip()

# ---------- EXCEL → CONVERSATIONS ----------
def convert_jawi_rumi_excels_to_conversations(
    input_xlsx_list,
    output_json,
    exclude_sheets=None
):
    if exclude_sheets is None:
        exclude_sheets = []

    conversations = []
    total_skipped = 0

    for input_xlsx in input_xlsx_list:
        print(f"\n📘 Reading file: {input_xlsx}")
        all_sheets = pd.read_excel(input_xlsx, sheet_name=None)

        for sheet_name, df in all_sheets.items():
            if sheet_name in exclude_sheets:
                print(f"⚠️ Skipping sheet: {sheet_name}")
                continue

            skipped_rows = []
            print(f"✅ Processing sheet: {sheet_name} ({len(df)} rows)")

            for idx, row in df.iterrows():
                jawi_text = clean_text(row.get("Jawi", ""))
                rumi_text = clean_text(row.get("Rumi", ""))

                if not jawi_text or not rumi_text:
                    skipped_rows.append(idx + 1)
                    continue

                # Jawi → Rumi
                conversations.append({
                    "conversations": [
                        {"role": "system", "content": "You are an expert AI that transliterates Classical Jawi into Rumi accurately."},
                        {"role": "user", "content": f"<jawi2rumi> {jawi_text}"},
                        {"role": "assistant", "content": rumi_text},
                    ]
                })

                # Rumi → Jawi
                conversations.append({
                    "conversations": [
                        {"role": "system", "content": "You are an expert AI that transliterates Rumi into Classical Jawi accurately."},
                        {"role": "user", "content": f"<rumi2jawi> {rumi_text}"},
                        {"role": "assistant", "content": jawi_text},
                    ]
                })

            if skipped_rows:
                total_skipped += len(skipped_rows)
                print(f"   ⚠️ Skipped {len(skipped_rows)} rows in '{sheet_name}'")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(conversations, f, ensure_ascii=False, indent=2)

    print(f"\n🔥 Done! {len(conversations)} examples saved → {output_json}")
    if total_skipped:
        print(f"⚠️ Total skipped rows: {total_skipped}")

OUTPUT_DIR = os.path.join(
    BASE_DIR, "jawiverse", "nirnama", "v1"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR, "bitransliteration_nirmala_model_v1.json"
)

convert_jawi_rumi_excels_to_conversations(
    input_xlsx_list=[
        os.path.join(BASE_DIR, "jawiverse", "nirnama", "dataset", "jawi_rumi_v1.xlsx"),
        os.path.join(BASE_DIR, "jawiverse", "nirnama", "dataset", "jawi_rumi_parallel_v1.xlsx"),
    ],
    output_json=OUTPUT_JSON,
)

# ---------- LOAD DATASET ----------
dataset = load_dataset(
    "json",
    data_files=OUTPUT_JSON,
    split="train"
)

# ---------- STANDARDIZE SHAREGPT (IMPORTANT) ----------
dataset = standardize_sharegpt(dataset)

# ---------- APPLY CHAT TEMPLATE ----------
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        for convo in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names
)

# ---------- QUICK SANITY CHECK ----------
print("✅ Dataset ready for training")
print("Sample conversations:\n", dataset[5]["text"])
dataset



📘 Reading file: /content/drive/MyDrive/jawiverse/nirnama/dataset/jawi_rumi_v1.xlsx
✅ Processing sheet: Sheet1 (13573 rows)

📘 Reading file: /content/drive/MyDrive/jawiverse/nirnama/dataset/jawi_rumi_parallel_v1.xlsx
✅ Processing sheet: Sheet1 (13573 rows)
   ⚠️ Skipped 2 rows in 'Sheet1'

🔥 Done! 54288 examples saved → /content/drive/MyDrive/jawiverse/nirnama/v1/bitransliteration_nirmala_model_v1.json
⚠️ Total skipped rows: 2


Generating train split: 0 examples [00:00, ? examples/s]

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/54288 [00:00<?, ? examples/s]

Map:   0%|          | 0/54288 [00:00<?, ? examples/s]

✅ Dataset ready for training
Sample conversations:
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are an expert AI that transliterates Rumi into Classical Jawi accurately.<|eot_id|><|start_header_id|>user<|end_header_id|>

<rumi2jawi> a<|eot_id|><|start_header_id|>assistant<|end_header_id|>

ا<|eot_id|>


Dataset({
    features: ['text'],
    num_rows: 54288
})

In [ ]:
import sys
sys.version

'3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]'

# 3: TRAINING (WITH CHECKPOINTS + DRIVE)

In [ ]:
import os
print(os.environ.get("TG_BOT_TOKEN"))


None


In [ ]:
import requests
import os
TOKEN = "<YOUR_TG_BOT_TOKEN>"
CHAT_ID = 123456789  # your user id

url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "🧪 Absolute hard test",
}

r = requests.post(url, json=payload)
print(r.status_code, r.text)


200 {"ok":true,"result":{"message_id":94,"from":{"id":8212723123,"is_bot":true,"first_name":"<redacted>","username":"<redacted>"},"chat":{"id":123456789,"first_name":"<redacted>","last_name":"<redacted>","username":"<redacted>","type":"private"},"date":1769131060,"text":"\ud83e\uddea Absolute hard test"}}


In [ ]:
# ===============================
# TRAINING (SEGMENTED CHECKPOINTS + STATUS PRINTS)
# ===============================
import math
import os
import torch
import requests
import time

from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint
from unsloth.chat_templates import train_on_responses_only

# --- TELEGRAM CONFIG ---
#TELEGRAM_BOT_TOKEN = os.environ.get("TG_BOT_TOKEN")
#TELEGRAM_CHAT_ID = os.environ.get("TG_CHAT_ID")
TELEGRAM_BOT_TOKEN = os.environ["TG_BOT_TOKEN"] = "<YOUR_TG_BOT_TOKEN>"
TELEGRAM_CHAT_ID = os.environ["TG_CHAT_ID"] = "123456789"

print("TOKEN:", bool(os.environ.get("TG_BOT_TOKEN")))
print("CHAT :", bool(os.environ.get("TG_CHAT_ID")))

def telegram_notify(message: str):
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "disable_web_page_preview": True,
    }

    for attempt in range(2):  # retry once
        try:
            requests.post(url, json=payload, timeout=5)
            return
        except Exception as e:
            if attempt == 0:
                time.sleep(1)
            else:
                print("⚠️ Telegram notify failed:", e)

from transformers import TrainerCallback

class TelegramETACallback(TrainerCallback):
    def __init__(
        self,
        segment_id,
        total_segments,
        segment_total_steps,
        notify_every_steps=200,
    ):
        self.segment_id = segment_id
        self.total_segments = total_segments
        self.segment_total_steps = segment_total_steps
        self.notify_every_steps = notify_every_steps
        self.start_time = None
        self.last_notified_step = 0

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        # current step INSIDE THIS SEGMENT
        segment_step = state.global_step % self.segment_total_steps

        if segment_step == 0:
            return

        if segment_step - self.last_notified_step < self.notify_every_steps:
            return

        elapsed = time.time() - self.start_time
        sec_per_step = elapsed / segment_step
        remaining_steps = self.segment_total_steps - segment_step
        eta_sec = remaining_steps * sec_per_step

        msg = (
            f"📊 Training update\n"
            f"Segment: {self.segment_id}/{self.total_segments}\n"
            f"Progress: {segment_step}/{self.segment_total_steps}\n"
            f"Elapsed: {elapsed/60:.1f} min\n"
            f"ETA remaining: {eta_sec/3600:.2f} hr\n"
            f"Speed: {1/sec_per_step:.2f} it/s"
        )

        telegram_notify(msg)
        self.last_notified_step = segment_step

def format_eta(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f} min"
    else:
        return f"{seconds/3600:.2f} hr"

# ---------- PATHS ----------
#OUTPUT_DIR = "/content/drive/MyDrive/jawiverse/nirnama/v1"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
FINAL_DIR = f"{OUTPUT_DIR}/final"

# ---------- ENSURE DIRS EXIST ----------
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

# ---------- STEP CALCULATION ----------
num_samples = len(dataset)

per_device_train_batch_size = 4
gradient_accumulation_steps = 8
num_train_epochs = 3

effective_batch = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = math.ceil(num_samples / effective_batch)
total_steps = steps_per_epoch * num_train_epochs

num_segments = 5   # change freely (4=quartiles, 5=quintiles, etc.)
save_steps = max(1, total_steps // num_segments)

print("\n📊 Training plan")
print(f"Samples            : {num_samples}")
print(f"Steps / epoch      : {steps_per_epoch}")
print(f"Total steps        : {total_steps}")
print(f"Segments           : {num_segments}")
print(f"Checkpoint every   : {save_steps} steps")

# ---------- TRAINER ----------
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=256,
    packing=False,

    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
    ),

    args=SFTConfig(
        output_dir=CHECKPOINT_DIR,

        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        warmup_ratio=0.05,
        learning_rate=5e-5,

        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        fp16=True,
        seed=3407,

        logging_steps=10,
        report_to="none",

        save_strategy="steps",
        save_steps=save_steps,
        save_total_limit=num_segments,
    )
)

# ---------- TRAIN ON RESPONSES ONLY ----------
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# ---------- GPU INFO ----------
gpu = torch.cuda.get_device_properties(0)
print(f"\n🖥 GPU: {gpu.name} | VRAM: {round(gpu.total_memory / 1024**3, 2)} GB")

# ---------- INITIAL ETA ESTIMATE ----------
# Conservative initial estimate (RTX 3090 + Unsloth typical range)
# Adjust if you have historical data
estimated_sec_per_step = 0.45   # safe default for LoRA on 3090
estimated_total_eta_sec = total_steps * estimated_sec_per_step

# ---------- CHECK TRAINING STATE ----------
last_ckpt = get_last_checkpoint(CHECKPOINT_DIR)

def is_valid_checkpoint(path):
          if path is None:
              return False
          files = set(os.listdir(path))
          return (
              "trainer_state.json" in files
              or "adapter_model.safetensors" in files
          )

# If final model already exists, stop early
if os.path.exists(os.path.join(FINAL_DIR, "adapter_model.safetensors")):
    print("\n🏁 Training already completed.")
    print(f"📦 Final model found at: {FINAL_DIR}")
    print("✅ No further training needed.")
else:
    if last_ckpt is None:
        current_step = 0

        msg = (
            "🆕 *Training started*\n"
            f"🧠 GPU: {gpu.name}\n"
            f"📊 Samples: {num_samples}\n"
            f"🔢 Total steps: {total_steps}\n"
            f"📦 Segments: {num_segments}\n"
            f"📐 Effective batch: {effective_batch}\n"
            f"⏳ Estimated total ETA: {format_eta(estimated_total_eta_sec)}\n"
            "_ETA will refine after first segment_"
        )

        print("\n" + msg)
        telegram_notify(msg)
    else:
        current_step = int(last_ckpt.split("-")[-1])
        remaining_steps = total_steps - current_step

        estimated_remaining_eta_sec = remaining_steps * estimated_sec_per_step

        msg = (
            "🔁 *Training resumed*\n"
            f"📦 Checkpoint: `{os.path.basename(last_ckpt)}`\n"
            f"🔢 Remaining steps: {remaining_steps}\n"
            f"⏳ Estimated remaining ETA: {format_eta(estimated_remaining_eta_sec)}"
        )

        print("\n" + msg)
        telegram_notify(msg)

    # Determine segment index
    current_segment = (current_step // save_steps) + 1
    target_step = min(current_step + save_steps, total_steps)

    print(
        f"🚀 Training segment {current_segment} / {num_segments} "
        f"(steps {current_step} → {target_step})"
    )

    telegram_notify(
        f"🚀 *Training segment started*\n"
        f"📦 Segment: {current_segment}/{num_segments}\n"
        f"🔢 Steps: {current_step} → {target_step}\n"
        f"🧠 GPU: {gpu.name}"
    )

    # ---------- TRAIN (TRL-SAFE SEGMENTED TRAINING) ----------
    steps_this_run = target_step - current_step

    # Set absolute max_steps for this run
    trainer.args.max_steps = current_step + steps_this_run

    trainer.add_callback(
        TelegramETACallback(
            segment_id=current_segment,
            total_segments=num_segments,
            segment_total_steps=steps_this_run,
            notify_every_steps=50,  # change if you want fewer messages
        )
    )

    start_time = time.time()

    try:
      resume_ckpt = last_ckpt if is_valid_checkpoint(last_ckpt) else None
      trainer.train(resume_from_checkpoint=resume_ckpt)
    except Exception as e:
        msg = (
            "❌ *Training crashed*\n"
            f"📦 Segment: {current_segment}/{num_segments}\n"
            f"💥 Error: `{str(e)[:300]}`"
        )
        telegram_notify(msg)
        raise

    end_time = time.time()
    elapsed_sec = end_time - start_time
    elapsed_min = elapsed_sec / 60

    steps_done = target_step - current_step
    sec_per_step = elapsed_sec / max(1, steps_done)

    remaining_steps = total_steps - target_step
    eta_next_sec = sec_per_step * min(save_steps, remaining_steps)
    eta_total_sec = sec_per_step * remaining_steps


    # ---------- AFTER SEGMENT ----------
    if target_step >= total_steps:
        trainer.save_model(FINAL_DIR)
        tokenizer.save_pretrained(FINAL_DIR)

        msg = (
            "🏁 *Training completed*\n"
            "📦 Final model saved\n"
            f"📍 Path: `{FINAL_DIR}`"
        )
        print("\n" + msg)
        telegram_notify(msg)
    else:
        completed_segment = current_segment
        msg = (
            f"⏸ *Segment completed*\n"
            f"📦 Segment: {completed_segment}/{num_segments}\n"
            f"➡️ Rerun notebook to continue"
        )
        print("\n" + msg)
        telegram_notify(msg)


TOKEN: True
CHAT : True

📊 Training plan
Samples            : 54288
Steps / epoch      : 1697
Total steps        : 5091
Segments           : 5
Checkpoint every   : 1018 steps


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/54288 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/54288 [00:00<?, ? examples/s]


🖥 GPU: Tesla T4 | VRAM: 14.74 GB

🔁 *Training resumed*
📦 Checkpoint: `checkpoint-4072`
🔢 Remaining steps: 1019
⏳ Estimated remaining ETA: 7.6 min
🚀 Training segment 5 / 5 (steps 4072 → 5090)


Step,Training Loss
4080,0.053100
4090,0.054700
4100,0.042500
4110,0.054200
4120,0.049800
4130,0.064600
4140,0.052700
4150,0.057300
4160,0.059900
4170,0.071200


# 3: Inference / Testing

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_PATH = "/content/drive/MyDrive/jawiverse_checkpoints/final"
# or:
# MODEL_PATH = "/content/drive/MyDrive/jawiverse_checkpoints/checkpoint-1200"

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_PATH,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

prompt = "[INST] Explain Jawiverse project in simple terms [/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
